In [ ]:
# Installing required packages
!pip install anthropic  # For Claude API
!pip install scikit-learn
!pip install pandas numpy
!pip install transformers torch
# Cloning FakeNewsNet repository
!git clone https://github.com/KaiDMML/FakeNewsNet.git

In [ ]:
from anthropic import Anthropic
from typing import List, Dict
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
import json
import os
import subprocess

Alternate way

In [ ]:
# Installs
!pip -q install --upgrade pip
!pip -q install "anthropic>=0.34" "pandas>=2.0" "numpy>=1.24" "scikit-learn>=1.3" "transformers>=4.40" "torch>=2.2"

# Claude API key

repo_url = "https://github.com/KaiDMML/FakeNewsNet.git"
repo_dir = "/content/FakeNewsNet"
if not os.path.exists(repo_dir):
    !git clone $repo_url
else:
    !git -C $repo_dir pull

DATA LOADING AND DATASET BALANCING

This cell contains the foundational data preparation functions for the FakeNewsNet experiment:

1. load_fakenewsnet_data():
   - Loads 4 CSV files (politifact_fake, politifact_real, gossipcop_fake, gossipcop_real)
   - Combines them into a single DataFrame with 23,196 articles
   - Standardizes column names and adds 'label' column (1=fake, 0=real)
   - Handles missing 'text' column by using 'title' as fallback

2. create_balanced_sample():
   - Addresses severe class imbalance (original: 5,755 fake vs 17,441 real)
   - Creates balanced dataset with equal fake/real articles
   - Default: 250 samples per class (500 total)
   - Ensures stratified sampling and shuffling
   - Reports distribution statistics by source

Output: Balanced dataset ready for experiments with 50/50 fake/real split

In [ ]:
def load_fakenewsnet_data(base_path='FakeNewsNet/dataset'):
    """
    Load articles from FakeNewsNet CSV files
    """
    all_data = []

    # Defining the CSV files and their corresponding labels
    csv_files = [
        ('politifact_fake.csv', 'politifact', 1),  # fake = 1
        ('politifact_real.csv', 'politifact', 0),  # real = 0
        ('gossipcop_fake.csv', 'gossipcop', 1),
        ('gossipcop_real.csv', 'gossipcop', 0)
    ]

    base_path = Path(base_path)

    for filename, source, label in csv_files:
        file_path = base_path / filename

        if file_path.exists():
            print(f"Loading {filename}...")
            try:
                # Loading the CSV file
                df_temp = pd.read_csv(file_path)

                # Adding source and label columns
                df_temp['source'] = source
                df_temp['label'] = label

                print(f"  - Loaded {len(df_temp)} articles from {filename}")
                print(f"  - Columns: {df_temp.columns.tolist()}")

                all_data.append(df_temp)
            except Exception as e:
                print(f"  - Error loading {filename}: {e}")
        else:
            print(f"  - File not found: {file_path}")

    if all_data:
        # Combining all dataframes
        df = pd.concat(all_data, ignore_index=True)
        print(f"\nCombined dataset columns: {df.columns.tolist()}")
        column_mappings = {
            'news_url': 'url',
            'URL': 'url',
            'Title': 'title',
            'tweet_ids': 'tweet_ids',
            'Text': 'text',
            'text': 'text',
            'title': 'title'
        }

        # Renaming columns to standard names
        df = df.rename(columns=column_mappings)

        if 'id' not in df.columns:
            df['id'] = df.index.astype(str)

        if 'text' not in df.columns and 'title' in df.columns:
            print("\nNote: No 'text' column found, using 'title' for text analysis")
            df['text'] = df['title']

        return df
    else:
        print("No data files were loaded successfully")
        return pd.DataFrame()

def create_balanced_sample(df, sample_size_per_class=250, random_state=42):
    """
    Create a balanced dataset with equal numbers of fake and real articles

    Parameters:
    - df: Original dataframe
    - sample_size_per_class: Number of samples for each class (fake/real)
    - random_state: Random seed for reproducibility

    Returns:
    - Balanced dataframe
    """
    # Counting available samples
    n_fake = (df['label'] == 1).sum()
    n_real = (df['label'] == 0).sum()

    print(f"\nOriginal dataset distribution:")
    print(f"  - Fake articles: {n_fake}")
    print(f"  - Real articles: {n_real}")

    # Adjusting sample size if not enough samples
    max_sample_size = min(n_fake, n_real, sample_size_per_class)

    if max_sample_size < sample_size_per_class:
        print(f"\n Requested {sample_size_per_class} samples per class, but only {max_sample_size} available")
        print(f"   Using {max_sample_size} samples per class instead")

    # Sample equal numbers of fake and real articles
    fake_articles = df[df['label'] == 1].sample(n=max_sample_size, random_state=random_state)
    real_articles = df[df['label'] == 0].sample(n=max_sample_size, random_state=random_state)

    # Combine and shuffle
    df_balanced = pd.concat([fake_articles, real_articles])
    df_balanced = df_balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print(f"\nBalanced dataset created:")
    print(f"  - Total samples: {len(df_balanced)}")
    print(f"  - Fake articles: {(df_balanced['label'] == 1).sum()} ({(df_balanced['label'] == 1).mean()*100:.1f}%)")
    print(f"  - Real articles: {(df_balanced['label'] == 0).sum()} ({(df_balanced['label'] == 0).mean()*100:.1f}%)")
    print(f"\nDistribution by source in balanced dataset:")
    for source in df_balanced['source'].unique():
        source_df = df_balanced[df_balanced['source'] == source]
        print(f"  {source}:")
        print(f"    - Fake: {(source_df['label'] == 1).sum()}")
        print(f"    - Real: {(source_df['label'] == 0).sum()}")

    return df_balanced

ZERO-SHOT CLAUDE CLASSIFICATION EXPERIMENT

This cell implements direct fake/real classification without feature extraction:

1. zero_shot_claude_classification():
   - Sends each headline directly to Claude API for classification
   - Prompt asks Claude to determine fake vs real based on:
     * Sensationalism, emotional manipulation, clickbait tactics
     * Professional tone and credibility indicators
   - Returns: predictions (0/1), confidence scores (0-1), reasoning text
   - Includes rate limiting (0.3s/request) and error handling

2. run_zero_shot_experiment():
   - Orchestrates complete experiment workflow
   - Samples data, runs classification, evaluates performance
   - Calculates metrics: accuracy, precision, recall, F1 score
   - Generates visualizations:
     * Confusion matrix heatmap
     * Confidence distributions (by label and correctness)
   - Shows 5 sample misclassifications with Claude's reasoning

3. Main execution:
   - Loads and balances data (500 articles)
   - Runs zero-shot classification
   - Reports final accuracy (~65.4% typical)
   - Compares with feature-based approaches


In [ ]:
# Zero-Shot Claude Classification Baseline Experiment
# ============================================
# ZERO-SHOT CLAUDE CLASSIFICATION
# ============================================
def zero_shot_claude_classification(df_sample, api_key, model="claude-3-haiku-20240307"):
    """
    Use Claude to directly classify articles as fake or real
    without any feature extraction - pure zero-shot classification
    """
    print("="*60)
    print("ZERO-SHOT CLAUDE CLASSIFICATION")
    print("="*60)

    # Initializig Claude client
    # client = Anthropic(api_key=api_key)

    predictions = []
    confidences = []
    reasons = []
    successful = 0
    failed = 0

    print(f"\nProcessing {len(df_sample)} articles for direct classification...")
    for idx, row in df_sample.iterrows():
        try:
            title = str(row.get('title', ''))

            # Creating zero-shot classification prompt
            prompt = f"""Analyze this news headline and determine if it appears to be fake news or real news.

Headline: {title}

Considering factors like:
- Sensationalism and exaggeration
- Emotional manipulation
- Clickbait tactics
- Professional tone
- Credibility indicators

Respond with ONLY a JSON object in this exact format:
{{
    "classification": "fake" or "real",
    "confidence": 0.0 to 1.0,
    "brief_reason": "One sentence explanation"
}}
"""

            # Get Claude's classification
            message = client.messages.create(
                model=model,
                max_tokens=150,
                temperature=0,
                messages=[{
                    "role": "user",
                    "content": prompt
                }]
            )

            response_text = message.content[0].text.strip()

            # Parse JSON response
            if '```json' in response_text.lower():
                response_text = response_text.split('```json')[-1].split('```')[0]
            elif '```' in response_text:
                response_text = response_text.split('```')[1] if '```' in response_text else response_text

            # Find JSON object
            start_idx = response_text.find('{')
            end_idx = response_text.rfind('}')
            if start_idx != -1 and end_idx != -1:
                json_str = response_text[start_idx:end_idx+1]
                result = json.loads(json_str)

                # Convert classification to binary
                if result['classification'].lower() == 'fake':
                    predictions.append(1)
                else:
                    predictions.append(0)

                confidences.append(float(result.get('confidence', 0.5)))
                reasons.append(result.get('brief_reason', 'No reason provided'))
                successful += 1
            else:
                raise ValueError("No JSON found in response")

            # Progress update
            if (successful + failed) % 20 == 0:
                print(f"Progress: {successful + failed}/{len(df_sample)} (✓{successful} ✗{failed})")

            # Rate limiting
            time.sleep(0.3)

        except Exception as e:
            failed += 1
            # Default to real (0) if classification fails
            predictions.append(0)
            confidences.append(0.5)
            reasons.append("Error in classification")

            if "rate_limit" in str(e).lower():
                print(f"Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)
            elif failed <= 5:
                print(f"Error for article {idx}: {str(e)[:100]}")

    print(f"\n Classification complete: {successful} successful, {failed} failed")
    print(f"Average confidence: {np.mean(confidences):.3f}")

    return np.array(predictions), np.array(confidences), reasons

# ============================================
# RUN ZERO-SHOT EXPERIMENT
# ============================================
def run_zero_shot_experiment(api_key, df_balanced, sample_size=500):
    """
    Run complete zero-shot classification experiment with evaluation
    """
    print("\n" + "="*70)
    print("ZERO-SHOT CLAUDE CLASSIFICATION EXPERIMENT")
    print("="*70)

    # Sample data
    df_sample = df_balanced.sample(n=min(sample_size, len(df_balanced)), random_state=42)
    true_labels = df_sample['label'].values

    print(f"\nDataset: {len(df_sample)} articles")
    print(f"  - Fake: {sum(true_labels)} ({sum(true_labels)/len(true_labels)*100:.1f}%)")
    print(f"  - Real: {len(true_labels)-sum(true_labels)} ({(1-sum(true_labels)/len(true_labels))*100:.1f}%)")

    # Run classification
    start_time = time.time()
    predictions, confidences, reasons = zero_shot_claude_classification(df_sample, api_key)
    elapsed_time = time.time() - start_time

    min_len = min(len(predictions), len(true_labels))
    predictions = predictions[:min_len]
    confidences = confidences[:min_len]
    true_labels = true_labels[:min_len]

    # Calculate metrics
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, zero_division=0)
    recall = recall_score(true_labels, predictions, zero_division=0)
    f1 = f1_score(true_labels, predictions, zero_division=0)

    # Print results
    print("\n" + "="*60)
    print("ZERO-SHOT CLASSIFICATION RESULTS")
    print("="*60)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Average Confidence: {np.mean(confidences):.3f}")
    print(f"Processing Time: {elapsed_time:.1f} seconds")
    print(f"Time per article: {elapsed_time/len(df_sample):.2f} seconds")

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(true_labels, predictions,
                              target_names=['Real', 'Fake'], zero_division=0))

    # Confusion Matrix
    cm = confusion_matrix(true_labels, predictions)
    print("\nConfusion Matrix:")
    print(f"True Negative (Real->Real): {cm[0,0]}")
    print(f"False Positive (Real->Fake): {cm[0,1]}")
    print(f"False Negative (Fake->Real): {cm[1,0]}")
    print(f"True Positive (Fake->Fake): {cm[1,1]}")

    # Visualize Confusion Matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted Real', 'Predicted Fake'],
                yticklabels=['Actual Real', 'Actual Fake'])
    plt.title('Zero-Shot Claude Classification - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    # Confidence Analysis
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.hist(confidences[true_labels == 0], alpha=0.5, label='Real News', bins=20)
    plt.hist(confidences[true_labels == 1], alpha=0.5, label='Fake News', bins=20)
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    plt.title('Confidence Distribution by True Label')
    plt.legend()

    plt.subplot(1, 2, 2)
    correct_predictions = predictions == true_labels
    plt.hist(confidences[correct_predictions], alpha=0.5, label='Correct', bins=20)
    plt.hist(confidences[~correct_predictions], alpha=0.5, label='Incorrect', bins=20)
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    plt.title('Confidence Distribution by Prediction Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

    # Sample of misclassifications
    print("\n" + "="*60)
    print("SAMPLE MISCLASSIFICATIONS")
    print("="*60)

    misclassified_indices = np.where(predictions != true_labels)[0]
    if len(misclassified_indices) > 0:
        print(f"\nShowing {min(5, len(misclassified_indices))} misclassified examples:")
        for i in misclassified_indices[:5]:
            print(f"\nTitle: {df_sample.iloc[i]['title']}")
            print(f"True Label: {'Fake' if true_labels[i] == 1 else 'Real'}")
            print(f"Predicted: {'Fake' if predictions[i] == 1 else 'Real'}")
            print(f"Confidence: {confidences[i]:.3f}")
            if i < len(reasons):
                print(f"Claude's Reasoning: {reasons[i]}")

    return predictions, confidences, accuracy

# ============================================
# MAIN EXECUTION
# ============================================
if __name__ == "__main__":

    # API_KEY = ""

    # Load and balance data
    print("Loading data...")
    df = load_fakenewsnet_data()
    df_balanced = create_balanced_sample(df, sample_size_per_class=250)

    # Run zero-shot experiment
    predictions, confidences, accuracy = run_zero_shot_experiment(
        api_key=API_KEY,
        df_balanced=df_balanced,
        sample_size=500
    )

    print("\n" + "="*70)
    print("ZERO-SHOT EXPERIMENT COMPLETED!")
    print("="*70)
    print(f"Final Accuracy: {accuracy:.4f}")

    # Comparing with your previous results
    print("\n" + "="*60)
    print("COMPARISON WITH YOUR PREVIOUS EXPERIMENTS:")
    print("="*60)
    print(f"Zero-Shot Claude: {accuracy:.4f}")
    print(f"Your Baseline (Claude features + LR): ~0.62")
    print(f"Your Experiment 2 (Combined features): ~0.68-0.72 (expected)")

**Zero-shot results summary**

**Key Metrics:**

Overall Accuracy: 65.0% - Better than random (50%) but room for improvement
Precision: 79.1% - When predicting "fake," correct 4 out of 5 times
Recall: 40.8% - Only catches 41% of actual fake news
F1 Score: 53.8% - Moderate balanced performance
Average Confidence: 79.6% - Claude is quite confident in its predictions

Critical Findings:
1. High False Negative Rate (59.2%)

Missed 148 out of 250 fake articles
Claude is too conservative, defaulting to "real" when uncertain
Problem: Vague or neutral headlines from unreliable sources classified as real

2. Strong Specificity (89.2%)

Correctly identified 223 out of 250 real articles
Only 27 false alarms (real marked as fake)
Claude is good at not falsely accusing legitimate news

3. Processing Efficiency:

1.08 seconds per article
100% success rate (no API failures)
Total time: ~9 minutes for 500 articles

Pattern Analysis from Misclassifications:
Why Fake News Was Missed:

Simple, factual-sounding headlines ("Leslie Jones (comedian)")
Celebrity news without obvious sensationalism
Headlines mimicking legitimate reporting style

Why Real News Was Misidentified:

Vague headlines ("Cindy Lou Who")
Lack of context made Claude suspicious